# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

## Google Drive Setup

The following cell mounts Google Drive and restores any previously saved `vocab.pkl` and
best model checkpoint so training can resume automatically across sessions.
If this is the first run, the files will be created and pushed to Drive after training.


In [ ]:
# ── Google Drive: mount and restore saved artifacts ───────────────────────
import os, shutil

GDRIVE_MOUNT  = '/content/drive'
GDRIVE_FOLDER = os.path.join(GDRIVE_MOUNT, 'MyDrive', 'cvnd_captioning')
VOCAB_FILE    = './vocab.pkl'
MODELS_DIR    = './models'
BEST_CKPT     = 'best_checkpoint.pt'

os.makedirs(MODELS_DIR, exist_ok=True)

def gdrive_push(local_path):
    """Copy a local file up to the project folder on Google Drive."""
    if not os.path.isdir(GDRIVE_FOLDER):
        return
    dst = os.path.join(GDRIVE_FOLDER, os.path.basename(local_path))
    shutil.copy2(local_path, dst)
    print(f'[GDrive] saved  {os.path.basename(local_path)} → {dst}')

def gdrive_pull(filename, local_path):
    """Restore a file from Google Drive. Returns True if found."""
    src = os.path.join(GDRIVE_FOLDER, filename)
    if os.path.exists(src):
        shutil.copy2(src, local_path)
        print(f'[GDrive] loaded {filename} from Drive')
        return True
    return False

# Mount Drive (works on Colab; no-op elsewhere)
try:
    from google.colab import drive
    drive.mount(GDRIVE_MOUNT)
    os.makedirs(GDRIVE_FOLDER, exist_ok=True)
    print(f'[GDrive] project folder: {GDRIVE_FOLDER}')
except ImportError:
    print('[GDrive] not on Colab – files will be saved locally only')

# Restore vocab (avoids re-building from COCO annotations every run)
_vocab_restored = gdrive_pull('vocab.pkl', VOCAB_FILE)

# Restore best checkpoint for optional resume
_ckpt_restored  = gdrive_pull(BEST_CKPT, os.path.join(MODELS_DIR, BEST_CKPT))


<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step. 
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file. 
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:** 
The architecture follows *Show, Attend and Tell* (Xu et al., 2015) — a CNN encoder with soft Bahdanau attention over spatial grid features feeding an LSTM decoder.

**Encoder (`EncoderCNN`):** A pretrained ResNet-50 backbone (ImageNet weights, top blocks frozen initially) with its `avgpool` and `fc` layers removed. An `AdaptiveAvgPool2d(14, 14)` then produces a 14×14 spatial feature map (196 locations × 2048 channels), which is linearly projected to `embed_size=512`. This gives the decoder 196 distinct spatial regions to attend over, dramatically richer than a single global feature vector.

**Attention (`BahdanauAttention`):** Additive (soft) attention computes an energy score for each of the 196 spatial locations using a 2-layer MLP over the encoder features and the previous LSTM hidden state. A softmax produces attention weights α, which are used to form a context vector as a weighted sum of encoder features. A sigmoid gate (β) further scales the context to suppress irrelevant locations (doubly-stochastic regularisation).

**Decoder (`DecoderRNN`):** A single `LSTMCell` whose input at each step is the concatenation of the current word embedding and the attended context vector. Hidden state h and cell c are initialised from the mean of the encoder features via learned linear projections. A fully-connected layer projects h to vocabulary logits.

Hyperparameter choices are grounded in Xu et al. (2015) and Vinyals et al. (2015): `embed_size=512`, `hidden_size=512`, `batch_size=64`, Adam optimizer with `lr=4e-4` for the decoder and `lr=1e-4` for encoder fine-tuning (enabled from epoch 2). `vocab_threshold=5` balances coverage vs. vocabulary size (~9 000–10 000 tokens).

References:
- Xu et al. (2015) *Show, Attend and Tell* https://arxiv.org/pdf/1502.03044.pdf
- Vinyals et al. (2015) *Show and Tell* https://arxiv.org/pdf/1411.4555.pdf

### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and 
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:** 
The base transform (Resize→RandomCrop→RandomHorizontalFlip→ToTensor→Normalize) is retained from the template because it already matches the ImageNet pre-processing expected by ResNet-50 (mean/std normalisation). Two additional augmentations were added: `ColorJitter` (brightness, contrast, saturation ±0.2) to make the encoder robust to lighting variation, and `RandomGrayscale(p=0.05)` to prevent over-reliance on colour as a captioning cue. `RandomCrop(224)` provides position jitter without distorting aspect ratio.

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters()) 
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:** 
Initially (epoch 1) only the decoder, attention module, and encoder projection head are trained — the ResNet backbone is frozen. This warm-up stabilises the attention weights before back-propagating through the convolutional layers. From epoch 2 onward, `encoder.fine_tune(True)` unlocks ResNet layers 2–4, allowing the backbone to adapt its feature representations to the captioning objective. Separate `Adam` parameter groups give the encoder a 4× smaller learning rate (`1e-4` vs `4e-4`) to preserve pretrained features while allowing gradual adaptation.

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:** 
Adam was chosen because it adapts learning rates per-parameter and converges reliably on LSTM-based seq2seq models. Two parameter groups are used so the encoder and decoder can have independent learning rates. `ReduceLROnPlateau` (factor=0.5, patience=1) halves the learning rate when epoch-average loss stops improving, providing automatic LR annealing without manual scheduling. Weight decay (`1e-4`) adds mild L2 regularisation to reduce overfitting.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import sys
sys.path.append('/opt/cocoapi/PythonAPI')
from pycocotools.coco import COCO
from data_loader import get_loader
from model import EncoderCNN, DecoderRNN
import math


## TODO #1: Select appropriate values for the Python variables below.
batch_size      = 64    # batch size — 64 balances GPU utilisation and memory on a T4/V100
vocab_threshold = 5     # minimum word count; words appearing <5 times become <unk>
vocab_from_file = _vocab_restored   # True if vocab.pkl was just restored from Drive
embed_size      = 512   # encoder projection + word-embedding dimensionality (Xu et al., 2015)
hidden_size     = 512   # LSTM hidden state size
num_epochs      = 5     # number of training epochs
save_every      = 1     # save encoder-N.pkl / decoder-N.pkl every epoch
print_every     = 100   # print running stats every 100 steps
log_file        = 'training_log.txt'   # per-step loss + perplexity log

# (Optional) TODO #2: Amend the image transform below.
# Kept the original Resize→RandomCrop→Flip→Normalize pipeline and added
# ColorJitter for lighting robustness and RandomGrayscale to prevent the
# model from over-relying on colour cues.
transform_train = transforms.Compose([
    transforms.Resize(256),                          # smaller edge resized to 256
    transforms.RandomCrop(224),                      # 224×224 crop from random position
    transforms.RandomHorizontalFlip(),               # horizontal flip p=0.5
    transforms.ColorJitter(                          # robustness to lighting/colour shifts
        brightness=0.2, contrast=0.2,
        saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),              # occasional grey image
    transforms.ToTensor(),                           # PIL → float tensor [0,1]
    transforms.Normalize((0.485, 0.456, 0.406),      # ImageNet mean/std for ResNet-50
                         (0.229, 0.224, 0.225))])

# Build data loader.
data_loader = get_loader(transform=transform_train,
                         mode='train',
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_from_file=vocab_from_file)

# The size of the vocabulary.
vocab_size = len(data_loader.dataset.vocab)

# Initialize the encoder and decoder.
encoder = EncoderCNN(embed_size)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size)

# Move models to GPU if CUDA is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder.to(device)

# Define the loss function.
# ignore_index=0 skips the <unk>/padding token so it does not dominate the gradient.
criterion = nn.CrossEntropyLoss(ignore_index=0).cuda() if torch.cuda.is_available() \
            else nn.CrossEntropyLoss(ignore_index=0)

# TODO #3: Specify the learnable parameters of the model.
# Epoch 1: train all decoder params + the encoder projection head only.
# The backbone weights are frozen; fine_tune(True) is called in the training
# loop after epoch 1 to unfreeze the upper ResNet blocks.
params = (list(decoder.parameters()) +
          [p for p in encoder.parameters() if p.requires_grad])

# TODO #4: Define the optimizer.
# Two Adam parameter groups: higher LR for the decoder, lower for the encoder
# (to preserve pretrained ResNet features while allowing gradual adaptation).
optimizer = torch.optim.Adam(
    [
        {'params': list(decoder.parameters()),
         'lr': 4e-4},
        {'params': [p for p in encoder.parameters() if p.requires_grad],
         'lr': 1e-4},
    ],
    weight_decay=1e-4,
)

# ReduceLROnPlateau: halve LR when epoch loss stops improving.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=1, verbose=True)

# Set the total number of training steps per epoch.
total_step = math.ceil(len(data_loader.dataset.caption_lengths) / data_loader.batch_sampler.batch_size)

<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

In [ ]:
import torch.utils.data as data
import numpy as np
import os
import requests
import time

# Open the training log file.
f = open(log_file, 'w')

old_time = time.time()
response = requests.request("GET",
                            "http://metadata.google.internal/computeMetadata/v1/instance/attributes/keep_alive_token",
                            headers={"Metadata-Flavor":"Google"})

# Track best epoch loss for saving the best model to Google Drive.
best_loss = float('inf')

for epoch in range(1, num_epochs+1):

    # ── Fine-tune encoder: unfreeze ResNet layer2–layer4 from epoch 2 ──────
    # During epoch 1 the backbone is frozen (warm-up for the attention head).
    # From epoch 2 onward upper ResNet blocks receive gradients at lr=1e-4.
    if epoch == 2:
        encoder.fine_tune(True)
        # Add newly unfrozen params to the encoder optimizer group
        optimizer.param_groups[1]['params'] = [
            p for p in encoder.parameters() if p.requires_grad]
        print('[Encoder] fine-tuning ON')

    epoch_losses = []

    for i_step in range(1, total_step+1):

        if time.time() - old_time > 60:
            old_time = time.time()
            requests.request("POST",
                             "https://nebula.udacity.com/api/v1/remote/keep-alive",
                             headers={'Authorization': "STAR " + response.text})

        # Randomly sample a caption length, and sample indices with that length.
        indices = data_loader.dataset.get_train_indices()
        # Create and assign a batch sampler to retrieve a batch with the sampled indices.
        new_sampler = data.sampler.SubsetRandomSampler(indices=indices)
        data_loader.batch_sampler.sampler = new_sampler

        # Obtain the batch.
        images, captions = next(iter(data_loader))

        # Move batch of images and captions to GPU if CUDA is available.
        images   = images.to(device)
        captions = captions.to(device)

        # Zero the gradients.
        decoder.zero_grad()
        encoder.zero_grad()

        # Pass the inputs through the CNN-RNN model.
        # encoder returns (B, 196, embed_size) — 196 spatial attention locations.
        features = encoder(images)
        # decoder returns (B, T-1, vocab_size) — predicts tokens at positions [1..T].
        outputs  = decoder(features, captions)

        # Calculate the batch loss.
        # Attention decoder predicts T-1 tokens: outputs[:,t,:] targets captions[:,t+1].
        # We align by slicing captions[:, 1:] as the target sequence.
        loss = criterion(outputs.view(-1, vocab_size),
                         captions[:, 1:].contiguous().view(-1))

        # Backward pass.
        loss.backward()

        # Gradient clipping prevents exploding gradients in the LSTM.
        nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=5.0)
        if epoch >= 2:   # clip encoder grads only when fine-tuning is on
            nn.utils.clip_grad_norm_(
                [p for p in encoder.parameters() if p.requires_grad], max_norm=5.0)

        # Update the parameters in the optimizer.
        optimizer.step()

        epoch_losses.append(loss.item())

        # Get training statistics.
        stats = 'Epoch [%d/%d], Step [%d/%d], Loss: %.4f, Perplexity: %5.4f' % (
            epoch, num_epochs, i_step, total_step,
            loss.item(), np.exp(loss.item()))

        # Print training statistics (on same line).
        print('\r' + stats, end="")
        sys.stdout.flush()

        # Print training statistics to file.
        f.write(stats + '\n')
        f.flush()

        # Print training statistics (on different line).
        if i_step % print_every == 0:
            print('\r' + stats)

    # ── End of epoch ────────────────────────────────────────────────────────
    avg_loss = float(np.mean(epoch_losses))
    print(f'\n[Epoch {epoch}] avg_loss={avg_loss:.4f}  '
          f'perplexity={np.exp(avg_loss):.2f}')

    # Step the LR scheduler based on epoch average loss.
    scheduler.step(avg_loss)

    # Save the weights.
    if epoch % save_every == 0:
        torch.save(decoder.state_dict(), os.path.join('./models', 'decoder-%d.pkl' % epoch))
        torch.save(encoder.state_dict(), os.path.join('./models', 'encoder-%d.pkl' % epoch))

    # Save best unified checkpoint and push to Google Drive.
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_ckpt_path = os.path.join(MODELS_DIR, BEST_CKPT)
        torch.save({
            'epoch':       epoch,
            'loss':        best_loss,
            'embed_size':  embed_size,
            'hidden_size': hidden_size,
            'vocab_size':  vocab_size,
            'encoder':     encoder.state_dict(),
            'decoder':     decoder.state_dict(),
        }, best_ckpt_path)
        gdrive_push(best_ckpt_path)
        print(f'[Best ✓] epoch {epoch}  loss={best_loss:.4f} saved to Drive')

# Close the training log file.
f.close()

# Push the training log to Drive for record keeping.
gdrive_push(log_file)

# Push vocab to Drive (in case this was the first build).
if not _vocab_restored and os.path.exists(VOCAB_FILE):
    gdrive_push(VOCAB_FILE)

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here. 

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.

In [ ]:
# (Optional) TODO: Validate your model.
# To compute BLEU-4 on the COCO validation split, uncomment and run the
# cell below.  Requires data_loader_val.py (see markdown above) and the
# coco-caption evaluation library.
#
# import json
# val_loader = get_loader(transform=transform_train, mode='train',
#                         batch_size=1, vocab_from_file=True,
#                         annotations_file='/opt/cocoapi/annotations/captions_val2014.json',
#                         img_folder='/opt/cocoapi/images/train2014/')
# results = []
# encoder.eval(); decoder.eval()
# for i, (_, image) in enumerate(val_loader):
#     image = image.to(device)
#     with torch.no_grad():
#         features = encoder(image)
#         output   = decoder.sample(features)
#     sentence = clean_sentence(output)  # defined in 3_Inference.ipynb
#     results.append({'image_id': val_loader.dataset.ids[i], 'caption': sentence})
# with open('captions_val2014_results.json', 'w') as fp:
#     json.dump(results, fp)
# print(f'Wrote {len(results)} captions to captions_val2014_results.json')